<a href="https://colab.research.google.com/github/eeeewyz/Audio-course/blob/main/8_ASR_whisper_longform_audio.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#用whisper进行Long-Form Transcription and Timestamps
#加载数据集
from datasets import load_dataset

dataset = load_dataset(
    "hf-internal-testing/librispeech_asr_dummy", "clean", split="validation"
)
dataset

README.md:   0%|          | 0.00/520 [00:00<?, ?B/s]

clean/validation-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 9.19MB            

clean/validation-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating validation split:   0%|          | 0/73 [00:00<?, ? examples/s]

Dataset({
    features: ['file', 'audio', 'text', 'speaker_id', 'chapter_id', 'id'],
    num_rows: 73
})

In [4]:
import torch
from transformers import pipeline

device = "cuda:0" if torch.cuda.is_available() else "cpu"
pipe = pipeline(
    "automatic-speech-recognition", model="openai/whisper-base", device=device
)

config.json:   0%|          | 0.00/1.98k [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  290MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/3.81k [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/494k [00:00<?, ?B/s]

normalizer.json:   0%|          | 0.00/52.7k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/34.6k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/2.19k [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/185k [00:00<?, ?B/s]

In [ ]:
# 为什么可以合并这些segments？构造长音频
# MLS 这类有声书数据，通常原始来源是一段较长的连续朗读，然后为了训练方便，被切成很多短 segment。比如原本可能是：

# 原始长录音：
# [A][B][C][D][E]...

# 数据集里变成：

# sample 0 = A
# sample 1 = B
# sample 2 = C
# sample 3 = D
# ...

In [5]:
import numpy as np

# 目标：构造一段约 5 分钟的长音频
target_length_in_m = 5

# 获取模型要求的采样率
sampling_rate = pipe.feature_extractor.sampling_rate

# 将 5 分钟转换成目标采样点数量：
# 分钟 × 60 = 秒
# 秒 × sampling_rate = 总采样点数
target_length_in_samples = target_length_in_m * 60 * sampling_rate

In [6]:
# 用于保存不断拼接起来的音频采样点
long_audio = []

# 遍历 dataset 中的每一个短音频样本
for sample in dataset:

    # sample["audio"]["array"] 是当前音频的 waveform
    # extend 会把当前 waveform 的所有采样点接到 long_audio 后面
    long_audio.extend(sample["audio"]["array"])

    # 当累计音频长度超过 5 分钟对应的采样点数量时停止
    if len(long_audio) > target_length_in_samples:
        break

# 将 Python list 转换成 NumPy array
# 方便后续送给 ASR pipeline
long_audio = np.asarray(long_audio)

In [7]:
# 根据采样点数量计算实际音频时长（秒）
seconds = len(long_audio) / 16000

# 将总秒数转换成：分钟 + 剩余秒数
minutes, seconds = divmod(seconds, 60)

# 输出最终拼接后的实际音频长度
print(f"Length of audio sample is {minutes} minutes {seconds:.2f} seconds")

Length of audio sample is 5.0 minutes 4.60 seconds


In [9]:
pipe(
    long_audio,
    max_new_tokens=256,
    generate_kwargs={"task": "transcribe"},
    chunk_length_s=30,
    batch_size=8,
)

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).


KeyboardInterrupt: 

In [ ]:
#若加上timestamps功能，加上这个参数：return_timestamps=True,
pipe(
    long_audio,
    max_new_tokens=256,
    generate_kwargs={"task": "transcribe"},
    chunk_length_s=30,
    batch_size=8,
    return_timestamps=True,
)["chunks"]

[transformers] Using `chunk_length_s` is very experimental with seq2seq models. The results will not necessarily be entirely accurate and will have caveats. More information: https://github.com/huggingface/transformers/pull/20104. Ignore this warning with pipeline(..., ignore_warning=True). To use Whisper for long-form transcription, use rather the model's `generate` method directly as the model relies on it's own chunking mechanism (cf. Whisper original paper, section 3.8. Long-form Transcription).
[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Both `max_new_tokens` (=256) and `max_length`(=448) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/mai